## Lecture 28

A robot moves in a 5×5 warehouse grid from (0,0) to (4,4). It must avoid unsafe cells and move exactly one cell up, down, left, or right at each step. Can it reach the goal in exactly 8 moves?

In [110]:
from z3 import Solver, sat, Int, And, Not, Or

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


In [111]:
N = 8
GRID_SIZE = 5

s = Solver()  

## Variables

In [112]:
robot_x = [Int(f'x_{i}') for i in range(N + 1)]
robot_y = [Int(f'y_{i}') for i in range(N + 1)]

## Start Position

In [113]:
s.add(robot_x[0] == 0, robot_y[0] == 0)
s

[x_0 == 0, y_0 == 0]

## Goal Position

In [114]:
s.add(robot_x[N] == 4, robot_y[N] == 4)
s

[x_0 == 0, y_0 == 0, x_8 == 4, y_8 == 4]

## Boundary

In [115]:
for i in range(N + 1):
    s.add(And(robot_x[i] >= 0, robot_x[i] <= GRID_SIZE - 1))
    s.add(And(robot_y[i] >= 0, robot_y[i] <= GRID_SIZE - 1))

s

[x_0 == 0,
 y_0 == 0,
 x_8 == 4,
 y_8 == 4,
 And(x_0 >= 0, x_0 <= 4),
 And(y_0 >= 0, y_0 <= 4),
 And(x_1 >= 0, x_1 <= 4),
 And(y_1 >= 0, y_1 <= 4),
 And(x_2 >= 0, x_2 <= 4),
 And(y_2 >= 0, y_2 <= 4),
 And(x_3 >= 0, x_3 <= 4),
 And(y_3 >= 0, y_3 <= 4),
 And(x_4 >= 0, x_4 <= 4),
 And(y_4 >= 0, y_4 <= 4),
 And(x_5 >= 0, x_5 <= 4),
 And(y_5 >= 0, y_5 <= 4),
 And(x_6 >= 0, x_6 <= 4),
 And(y_6 >= 0, y_6 <= 4),
 And(x_7 >= 0, x_7 <= 4),
 And(y_7 >= 0, y_7 <= 4),
 And(x_8 >= 0, x_8 <= 4),
 And(y_8 >= 0, y_8 <= 4)]

## Obstacle

In [116]:
obstacles = [(1, 1), (1, 2), (2, 1), (3, 3)]

for obs in obstacles:
    for i in range(N + 1):
        s.add(Not(And(robot_x[i] == obs[0], robot_y[i] == obs[1])))

## Movement

In [117]:
for i in range(N):
    s.add(
        Or(
            And(robot_x[i + 1] == robot_x[i] + 1, robot_y[i + 1] == robot_y[i]),
            And(robot_x[i + 1] == robot_x[i] - 1, robot_y[i + 1] == robot_y[i]),
            And(robot_x[i + 1] == robot_x[i], robot_y[i + 1] == robot_y[i] - 1),
            And(robot_x[i + 1] == robot_x[i], robot_y[i + 1] == robot_y[i] + 1)
        )
    )

## Solution

In [118]:
path = []

if s.check() == sat:
    m = s.model()

    print(f"Solution:")
    for i in range(N + 1):
        print(f"Step {i}: ({m[robot_x[i]]}, {m[robot_y[i]]})")
        path.append((m[robot_x[i]].as_long(), m[robot_y[i]].as_long()))
    
else:
    print("No solution exists.")



Solution:
Step 0: (0, 0)
Step 1: (0, 1)
Step 2: (0, 2)
Step 3: (0, 3)
Step 4: (1, 3)
Step 5: (1, 4)
Step 6: (2, 4)
Step 7: (3, 4)
Step 8: (4, 4)


## Visualization

The visualization code is generated using ChatGPT.

In [ ]:
import tkinter as tk

GRID_SIZE = 5
CELL_SIZE = 200

root = tk.Tk()
root.title("Robot Movement")

canvas = tk.Canvas(
    root,
    width=GRID_SIZE * CELL_SIZE,
    height=GRID_SIZE * CELL_SIZE
)
canvas.pack()

# Draw checkerboard
for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        color = "white" if (row + col) % 2 == 0 else "lightgray"

        canvas.create_rectangle(
            col * CELL_SIZE,
            row * CELL_SIZE,
            (col + 1) * CELL_SIZE,
            (row + 1) * CELL_SIZE,
            fill=color,
            outline="black"
        )

# Draw obstacles
for x, y in obstacles:
    canvas.create_rectangle(
        x * CELL_SIZE + 8,
        y * CELL_SIZE + 8,
        (x + 1) * CELL_SIZE - 8,
        (y + 1) * CELL_SIZE - 8,
        fill="red"
    )

# Mark start and goal
def center(position):
    x, y = position
    return (
        x * CELL_SIZE + CELL_SIZE // 2,
        y * CELL_SIZE + CELL_SIZE // 2
    )

sx, sy = center(path[0])
gx, gy = center(path[-1])

canvas.create_oval(
    sx - 12, sy - 12, sx + 12, sy + 12,
    fill="green"
)

canvas.create_oval(
    gx - 12, gy - 12, gx + 12, gy + 12,
    fill="gold"
)

# Create the robot
rx, ry = center(path[0])

robot = canvas.create_oval(
    rx - 15, ry - 15, rx + 15, ry + 15,
    fill="blue"
)

move_index = 0

def move_robot():
    global move_index

    if move_index < len(path) - 1:
        move_index += 1

        x, y = center(path[move_index])

        canvas.coords(
            robot,
            x - 15, y - 15,
            x + 15, y + 15
        )

        root.title(
            f"Robot Movement - Move {move_index}/{len(path) - 1}"
        )

        root.after(500, move_robot)

# Start animation after 1 second
root.after(1000, move_robot)

root.mainloop()

## New Obstacle

In [120]:
new_obstacle = (2, 2)
obstacles.append(new_obstacle)

for i in range(N + 1):
    s.add(Not(And(robot_x[i] == new_obstacle[0], robot_y[i] == new_obstacle[1])))

## All Possible Paths

In [121]:
count = 0

while s.check() == sat:
    count += 1
    m = s.model()
    print(f"Solution {count}:")
    for i in range(N + 1):
        print(f"Step {i}: ({m[robot_x[i]]}, {m[robot_y[i]]})")

    s.add(Not(And([robot_x[i] == m[robot_x[i]] and robot_y[i] == m[robot_y[i]] for i in range(N + 1)]))) 
else:
    print("No solution exists.")

Solution 1:
Step 0: (0, 0)
Step 1: (0, 1)
Step 2: (0, 2)
Step 3: (0, 3)
Step 4: (0, 4)
Step 5: (1, 4)
Step 6: (2, 4)
Step 7: (3, 4)
Step 8: (4, 4)
Solution 2:
Step 0: (0, 0)
Step 1: (0, 1)
Step 2: (0, 2)
Step 3: (0, 3)
Step 4: (1, 3)
Step 5: (2, 3)
Step 6: (2, 4)
Step 7: (3, 4)
Step 8: (4, 4)
Solution 3:
Step 0: (0, 0)
Step 1: (0, 1)
Step 2: (0, 2)
Step 3: (0, 3)
Step 4: (1, 3)
Step 5: (1, 4)
Step 6: (2, 4)
Step 7: (3, 4)
Step 8: (4, 4)
Solution 4:
Step 0: (0, 0)
Step 1: (1, 0)
Step 2: (2, 0)
Step 3: (3, 0)
Step 4: (3, 1)
Step 5: (3, 2)
Step 6: (4, 2)
Step 7: (4, 3)
Step 8: (4, 4)
Solution 5:
Step 0: (0, 0)
Step 1: (1, 0)
Step 2: (2, 0)
Step 3: (3, 0)
Step 4: (4, 0)
Step 5: (4, 1)
Step 6: (4, 2)
Step 7: (4, 3)
Step 8: (4, 4)
Solution 6:
Step 0: (0, 0)
Step 1: (1, 0)
Step 2: (2, 0)
Step 3: (3, 0)
Step 4: (3, 1)
Step 5: (4, 1)
Step 6: (4, 2)
Step 7: (4, 3)
Step 8: (4, 4)
No solution exists.
